# inflow-ledger: what the archive holds

Observed daily reservoir inflow (CDEC) with the newest archived HEFS ensemble as a fan, then the score table.
Run `python archive.py && python score.py` from the repo root first, or check out the `data` branch into `data/`.

In [ ]:
import sys
from pathlib import Path

import matplotlib.dates as mdates
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.ticker import FuncFormatter, LogLocator, NullFormatter

sys.path.insert(0, str(Path.cwd().parent))
from score import LEADS, SCORED, read_hefs, read_obs

DATA = Path.cwd().parent / "data"
NAMES = {"LAMC1": "Mendocino", "ORDC1": "Oroville", "SHDC1": "Shasta", "FOLC1": "Folsom", "NBBC1": "New Bullards Bar",
         "NMSC1": "New Melones", "FRAC1": "Millerton", "EXQC1": "McClure", "ISAC1": "Isabella", "TMDC1": "Kaweah",
         "SCSC1": "Success"}
INK, INK2, GRID = "#0b0b0b", "#52514e", "#e8e7e3"
BLUE, ORANGE, AQUA, VIOLET, MAGENTA, GREEN = "#2a78d6", "#eb6834", "#1baf7a", "#6250d6", "#e87ba4", "#008300"
plt.rcParams.update({"font.size": 9, "axes.titlesize": 10, "axes.titleweight": "medium", "axes.titlelocation": "left",
                     "axes.edgecolor": GRID, "axes.labelcolor": INK2, "xtick.color": INK2, "ytick.color": INK2,
                     "axes.spines.top": False, "axes.spines.right": False, "axes.grid": True, "axes.grid.axis": "y",
                     "grid.color": GRID, "grid.linewidth": 0.8, "xtick.major.size": 0, "ytick.major.size": 0,
                     "legend.frameon": False, "figure.facecolor": "white", "axes.facecolor": "white"})
flow_fmt = FuncFormatter(lambda v, _: f"{v / 1000:g}K" if v >= 10_000 else f"{v:,.0f}")


In [ ]:
def latest_fan(lid):
    """Percentiles per forecast day from the newest archived ensemble, in cfs, plus the issue timestamp."""
    path = sorted((DATA / "hefs_daily" / lid).glob("*.gz"))[-1]
    fc = read_hefs(path)
    q = {d: np.percentile(m, [5, 25, 50, 75, 95]) for d, m in fc.items()}
    return pd.DataFrame(q, index=["p05", "p25", "p50", "p75", "p95"]).T.iloc[: LEADS.stop], pd.Timestamp(path.stem[:8])

fig, axes = plt.subplots(3, 4, figsize=(16, 9.5))
for ax, (lid, sta) in zip(axes.flat, SCORED.items()):
    obs = pd.Series(read_obs(DATA / "cdec_inflow" / f"{sta}.csv")).sort_index()
    fan, issued = latest_fan(lid)
    start = fan.index[0] - pd.Timedelta(days=30)
    recent = obs[obs.index >= start]
    ax.fill_between(fan.index, fan.p05, fan.p95, color=BLUE, alpha=0.12, lw=0, label="HEFS 5-95%")
    ax.fill_between(fan.index, fan.p25, fan.p75, color=BLUE, alpha=0.28, lw=0, label="HEFS 25-75%")
    ax.plot(fan.index, fan.p50, color=BLUE, lw=2, solid_capstyle="round", label="HEFS median")
    ax.plot(recent.index, recent.values, color=INK, lw=2, solid_capstyle="round", label="Observed (CDEC)")
    ax.axvline(fan.index[0], color=INK2, lw=0.8)
    ax.set_yscale("log")
    lo, hi = max(1.0, 0.7 * min(recent.min(), fan.p05.min())), 1.3 * max(recent.max(), fan.p95.max())
    ax.set_ylim(lo, hi)
    ax.yaxis.set_major_locator(LogLocator(base=10, subs=(1, 2, 5)))
    ax.yaxis.set_major_formatter(flow_fmt)
    ax.yaxis.set_minor_formatter(NullFormatter())
    ax.set_xlim(start, fan.index[-1])
    ax.xaxis.set_major_locator(mdates.DayLocator(interval=7))
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%b %-d"))
    ax.set_title(f"{NAMES[lid]}  ·  {lid} / {sta}")
    ax.text(fan.index[0], hi, " forecast", color=INK2, fontsize=8, va="top")
for ax in axes.flat[len(SCORED):]:
    ax.axis("off")
axes[0, 0].set_ylabel("inflow, cfs (log)")
handles, labels = axes[0, 0].get_legend_handles_labels()
fig.legend(handles, labels, loc="upper right", ncol=4, bbox_to_anchor=(0.99, 0.995))
fig.suptitle(f"Reservoir inflow: 30 days observed and the newest HEFS ensemble, issued {issued:%b %-d, %Y} UTC",
             x=0.01, ha="left", fontsize=12, fontweight="medium", color=INK)
fig.tight_layout(rect=(0, 0, 1, 0.96))


In [ ]:
scores = pd.read_csv(DATA / "scores" / "inflow_daily.csv", parse_dates=["issue"])
print(f"{len(scores)} scored forecast-days, {scores.issue.nunique()} issue days, {scores.lid.nunique()} points")
if len(scores):
    by_lead = scores.groupby(["product", "lead"])[["crps", "crps_persistence", "crps_climatology", "in90"]].mean()
    display(by_lead.round(1))
    fig, ax = plt.subplots(figsize=(8, 4))
    hefs = by_lead.loc["hefs"]
    series = [(hefs["crps_persistence"], ORANGE, "persistence"), (hefs["crps_climatology"], AQUA, "climatology")]
    series += [(by_lead.loc[p, "crps"], c, p) for p, c in zip(by_lead.index.unique(0), [BLUE, VIOLET, MAGENTA, GREEN])]
    for line, color, name in series:
        ax.plot(line.index, line.values, color=color, lw=2, label=name)
        ax.text(line.index[-1] + 0.2, line.iloc[-1], name, color=INK2, va="center", fontsize=8)
    ax.yaxis.set_major_formatter(flow_fmt)
    ax.set(xlabel="lead, days", ylabel="mean CRPS, cfs", title="Pooled over scored reservoirs: lower is better")
    ax.legend(loc="upper left")
